# STIC validation on Ameriflux sites (version v1.4)

Validation of STIC using field data from the ameriflux database and landsat acquisitions

In [ ]:
import os

os.environ["NUMBA_DISABLE_JIT"] = "1"
# os.environ["NUMBA_DEVELOPER_MODE"] = "1"
# os.environ["NUMBA_DEBUG"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd

In [ ]:
import extra.plot as plt
from extra.metrics import generate_metrics_table
from extra.process import analyze, init_stic, run_stic

## Input

In [ ]:
data_path = os.path.join(
    os.path.expanduser("~"), "data", "validation", "fl_data.csv"
)

## Read data

In [ ]:
data = pd.read_csv(data_path, parse_dates=["date", "date (utc)"])

In [ ]:
data

Clean land cover classes

In [ ]:
# Remove SNO
data = data[data["landcover"] != "SNO"]
# Combine SAV with WSA
data.loc[data["landcover"] == "SAV", "landcover"] = "WSA"
# Combine BSV with OSH
data.loc[data["landcover"] == "BSV", "landcover"] = "OSH"

### Plot data

In [ ]:
plt.plot_data(data)

In [ ]:
plt.plot_scatter(data)

## Compute initialization of STIC model

In [ ]:
%%time
init_v1_4 = init_stic(data, version="1.4")
init_v1_4.attrs["label"] = "init STIC 1.4"

In [ ]:
analyze(init_v1_4)

In [ ]:
init_v1_4.head()

In [ ]:
plt.plot_scatter_with_colorbar(init_v1_4)

In [ ]:
plt.plot_scatter_3d(init_v1_4)

## Compute STIC model

In [ ]:
%%time
res_v1_4 = run_stic(data, version="1.4")

In [ ]:
res_v1_4["swir index"] = res_v1_4["nir/swir"]
res_v1_4["gs/gsmax"] = res_v1_4["gs"] / res_v1_4.groupby("landcover")[
    "gs"
].transform("max")
res_v1_4["ec_gs/gsmax"] = res_v1_4["ec_gs"] / res_v1_4.groupby("landcover")[
    "ec_gs"
].transform("max")

In [ ]:
plt.plot_scatter_3d(res_v1_4)

In [ ]:
res_v1_4

In [ ]:
stats = res_v1_4.describe()
stats["gs/ga"]

In [ ]:
plt.plot_scatter_with_colorbar(res_v1_4, x="ec_le_closed", y="le")

In [ ]:
plt.plot_scatter(res_v1_4, x="ec_le_closed", y="le")

In [ ]:
res_v1_4_stressed = res_v1_4[res_v1_4["stressed"] == 1]
res_v1_4_stressed.attrs["label"] = "stressed"
res_v1_4_unstressed = res_v1_4[res_v1_4["stressed"] == 0]
res_v1_4_unstressed.attrs["label"] = "unstressed"

In [ ]:
plt.plot_scatter(
    [res_v1_4_stressed, res_v1_4_unstressed],
    x="ec_le_closed",
    y="le",
    xrange=[0, 1000],
    yrange=[0, 1000],
    use_colors=False,
    font_size=12,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    [res_v1_4_stressed, res_v1_4_unstressed],
    x="ec_le_closed",
    y="le",
    c="ts-ta",
    xrange=[0, 1000],
    yrange=[0, 1000],
    use_stressed=True,
    marker_size=6,
    font_size=12,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    [res_v1_4_stressed, res_v1_4_unstressed],
    x="ec_le_closed",
    y="le",
    c="swir index",
    xrange=[0, 1000],
    yrange=[0, 1000],
    use_stressed=True,
    marker_size=6,
    font_size=12,
)

## Compute metrics

In [ ]:
metrics_v1_4 = generate_metrics_table(
    res_v1_4,
    variables=["le", "le_closed"],
    ranges={
        "le": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
        "le_closed": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
    },
)

In [ ]:
metrics_v1_4

In [ ]:
plt.plot_metrics(metrics_v1_4, "le_closed", "rmse", only_all=True)

In [ ]:
plt.plot_metrics(metrics_v1_4, "le_closed", "nrmse_mean")

In [ ]:
metrics_v1_4_stressed = generate_metrics_table(
    res_v1_4[res_v1_4["stressed"] == 1],
    variables=["le", "le_closed"],
    ranges={
        "le": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
        "le_closed": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
    },
)
metrics_v1_4_stressed.attrs["label"] = "stressed"
metrics_v1_4_unstressed = generate_metrics_table(
    res_v1_4[res_v1_4["stressed"] == 0],
    variables=["le", "le_closed"],
    ranges={
        "le": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
        "le_closed": [
            (float("-inf"), 200),
            (200, 400),
            (400, 600),
            (600, float("inf")),
        ],
    },
)
metrics_v1_4_unstressed.attrs["label"] = "unstressed"

In [ ]:
plt.plot_metrics(
    [metrics_v1_4_unstressed, metrics_v1_4_stressed],
    "le",
    "rmse",
    only_all=True,
)

In [ ]:
plt.plot_metrics([metrics_v1_4_unstressed, metrics_v1_4_stressed], "le", "rmse")

In [ ]:
plt.plot_metrics(
    [metrics_v1_4_unstressed, metrics_v1_4_stressed], "le_closed", "nrmse_mean"
)

## Error analysis

In [ ]:
res_v1_4.columns

In [ ]:
plt.analyze(
    res_v1_4,
    x="ts-ta",
    y="diff_le_closed",
    c="fc",
    bins=20,
    x_label="LST - Ta (°C)",
    y_label="Error in LE (W/m2)",
    color_label="Veg. cover",
    title="All landcover classes",
    width=900,
    height=600,
    x_range=[-20, 35],
    y_range=[-600, 600],
    c_range=[0, 1],
    font_size=12,
)

In [ ]:
plt.analyze(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ts-ta",
    y="diff_le_closed",
    c="fc",
    bins=10,
    x_label="LST - Ta (°C)",
    y_label="Error in LE (W/m2)",
    color_label="Veg. cover",
    title=None,
    width=800,
    height=500,
    x_range=[-20, 30],
    y_range=[-300, 300],
    c_range=[0, 1],
    font_size=12,
)


In [ ]:
plt.analyze(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ts-ta",
    y="diff_le_closed",
    c="fc",
    bins=10,
    x_label="LST - Ta (°C)",
    y_label="Error in LE (W/m2)",
    color_label="Veg. cover",
    title=None,
    width=800,
    height=500,
    x_range=[-20, 30],
    y_range=[-400, 300],
    c_range=[0, 1],
    font_size=12,
)

## Graphs

In [ ]:
res_v1_4.columns

In [ ]:
res_v1_4[["vari_green", "swir index", "ts-ta"]].describe()

In [ ]:
res_v1_4[["vari_green", "swir index", "ts-ta"]].dtypes

In [ ]:
res_v1_4.groupby("landcover")[["vari_green", "swir index", "ts-ta"]].agg(
    ["min", "max"]
)

1. x-axis: Ts-Ta; primary y-axis: gs/gmax (0 - 1) EC tower; secondary y-axis: SWIR index [ (NIR-SWIR)/(NIR + SWIR)]

In [ ]:
res_v1_4["1-ec_gs/gsmax"] = 1 - res_v1_4["ec_gs/gsmax"]

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ts-ta",
    y="1-ec_gs/gsmax",
    c="swir index",
    xrange=[-20, 30],
    yrange=[0.2, 1],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=500,
    width=600,
)


In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ts-ta",
    y="1-ec_gs/gsmax",
    c="swir index",
    xrange=[-20, 30],
    yrange=[0.2, 1],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=500,
    width=600,
)

2. x-axis: Ts-Ta; primary y-axis: gs/gmax (0 - 1) EC tower; secondary y-axis: vari_green

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ts-ta",
    y="1-ec_gs/gsmax",
    c="vari_green",
    xrange=[-20, 30],
    yrange=[0.2, 1],
    crange=[-0.3, 0.8],
    marker_size=8,
    font_size=12,
    height=500,
    width=600,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ts-ta",
    y="1-ec_gs/gsmax",
    c="vari_green",
    xrange=[-20, 30],
    yrange=[0.2, 1],
    crange=[-0.3, 0.8],
    marker_size=8,
    font_size=12,
    height=500,
    width=600,
)

3. x-axis: gs/ga from EC tower (0 - 1); y-axis: gs/ga from STIC (0 - 1); color coding by ts-ta, SWIR index

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ec_gs/gsmax",
    y="gs/gsmax",
    c="swir index",
    xrange=[0, 0.5],
    yrange=[0, 0.5],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=550,
    width=600,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ec_gs/gsmax",
    y="gs/gsmax",
    c="swir index",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=550,
    width=600,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ec_gs/gsmax",
    y="gs/gsmax",
    c="ts-ta",
    xrange=[0, 0.5],
    yrange=[0, 0.5],
    crange=[-20, 30],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ec_gs/gsmax",
    y="gs/gsmax",
    c="ts-ta",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-20, 30],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)

x-axis: EF from EC tower; y-axis: EF from STIC; color coding by SWIR index and ts-ta

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ec_ef",
    y="ef",
    c="swir index",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)


In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ec_ef",
    y="ef",
    c="swir index",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-0.3, 1],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "WSA"],
    x="ec_ef",
    y="ef",
    c="ts-ta",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-20, 30],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)

In [ ]:
plt.plot_scatter_with_colorbar(
    res_v1_4[res_v1_4["landcover"] == "CRO"],
    x="ec_ef",
    y="ef",
    c="ts-ta",
    xrange=[0, 1],
    yrange=[0, 1],
    crange=[-20, 30],
    marker_size=8,
    font_size=12,
    height=550,
    width=550,
)